# 02 · Entrenar (runtime **GPU**)
Menu: *Entorno de ejecucion → Cambiar tipo → GPU (T4)*. Cuenta: Gmail con el plan.
Antes: en tu PC `python tools/apex.py sync` (codigo) y haber corrido el notebook 01 (datos).
- Cambia `RUN` y `CONFIG` para otra corrida. Los checkpoints se guardan en Drive cada epoca.
- Si Colab se desconecta: vuelve a ejecutar las celdas 2, 3 y 4 y **reanuda sola**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/UTEC/DEEP'
import os, subprocess
assert os.path.isdir(DRIVE + '/apex_kart_code'), "Falta el codigo en Drive: en tu PC ejecuta  python tools/apex.py sync"
!rm -rf /content/apex && cp -r {DRIVE}/apex_kart_code /content/apex
%cd /content/apex

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!mkdir -p /content/processed_v1
!rsync -a --info=progress2 {DRIVE}/apex_kart_processed/v1/ /content/processed_v1/
!du -sh /content/processed_v1 && ls /content/processed_v1 | wc -l

In [ ]:
RUN = 'bc_v1'
CONFIG = 'configs/train/bc_v1.json'
!python training/train_bc.py --config {CONFIG} --data /content/processed_v1 \
    --out {DRIVE}/apex_kart_runs/{RUN} --max-minutes 150

In [ ]:
import json
for r in json.load(open(f'{DRIVE}/apex_kart_runs/{RUN}/log.json')):
    v = r.get('val', {})
    print(r['epoch'], 'train', round(r['train_loss'], 4), 'val', round(v.get('loss', float('nan')), 4),
          'steer', round(v.get('steer_huber', float('nan')), 4), r['samples_per_s'], 'muestras/s')
drive.flush_and_unmount(); print('Listo. En tu PC: python tools/apex.py pull', RUN)